# MLPipeline: NLP Sentiment Classification

Complete walkthrough of the MLPipeline project - from data preprocessing to model serving.

## Overview

This notebook demonstrates:
1. Text preprocessing for NLP
2. Model training with HuggingFace Transformers
3. Model evaluation and metrics
4. Inference on new data
5. Integration with Kubernetes and FastAPI

## Section 1: Set Up Development Environment

Configure dependencies and initialize the environment

In [1]:
# Cell 1: Set Up Development Environment
# Import required libraries
import sys
import os
from pathlib import Path

# Add project to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Standard library imports
import json
from typing import List, Dict

print("✓ Development environment configured")
print(f"Python version: {sys.version}")
print(f"Project root: {project_root}")
print(f"Current working directory: {os.getcwd()}")
print(f"\nProject structure verified: {project_root.exists()}")

✓ Development environment configured
Python version: 3.11.15 (main, Mar  3 2026, 09:26:23) [GCC 11.4.0]
Project root: /home/rongoodman/Projects/MLPipeline
Current working directory: /home/rongoodman/Projects/MLPipeline/notebooks

Project structure verified: True


## Section 2: Text Preprocessing for NLP

Demonstrating text cleaning and preprocessing pipeline

In [2]:
# Cell 2: Text Preprocessing Examples
from src.preprocessing.text_cleaning import clean_text, preprocess_batch

# Sample texts with various issues
sample_texts = [
    "Check out this amazing product: https://example.com! 😊 #awesome",
    "I really enjoyed this film! Great quality and fast delivery!!!",
    "This was disappointing... NOT as described. Very poor quality.",
    "It's okay, nothing special. Average product.",
    "BEST PURCHASE EVER!!! Highly recommended!!!!"
]

print("Original Texts:")
print("=" * 70)
for i, text in enumerate(sample_texts, 1):
    print(f"{i}. {text}")

# Apply preprocessing
print("\n\nCleaned Texts:")
print("=" * 70)
cleaned_texts = preprocess_batch(sample_texts, clean=True)
for i, text in enumerate(cleaned_texts, 1):
    print(f"{i}. {text}")

print("\n✓ Text preprocessing demonstrated")

Original Texts:
1. Check out this amazing product: https://example.com! 😊 #awesome
2. I really enjoyed this film! Great quality and fast delivery!!!
3. This was disappointing... NOT as described. Very poor quality.
4. It's okay, nothing special. Average product.
5. BEST PURCHASE EVER!!! Highly recommended!!!!


Cleaned Texts:
1. check out this amazing product 😊 awesome
2. i really enjoyed this film great quality and fast delivery
3. this was disappointing not as described very poor quality
4. its okay nothing special average product
5. best purchase ever highly recommended

✓ Text preprocessing demonstrated


## Section 3: Load Configuration

Load and review training/inference configuration

In [3]:
# Step 5: Load and Display Configuration Files
# Resolve config file paths relative to project root
from pathlib import Path
import yaml

# Get project root (parent of notebooks directory)
notebook_dir = Path.cwd()
project_root = notebook_dir.parent
config_dir = project_root / "configs"

# Load configuration files
def load_config_file(config_path):
    """Load YAML config file"""
    if not config_path.exists():
        raise FileNotFoundError(f"Configuration file not found: {config_path}")
    with open(config_path) as f:
        return yaml.safe_load(f)

training_config_path = config_dir / "training_config.yaml"
inference_config_path = config_dir / "inference_config.yaml"

training_config = load_config_file(training_config_path)
inference_config = load_config_file(inference_config_path)

print("Training Configuration:")
print("=" * 70)
print(json.dumps(training_config, indent=2))

print("\n\nInference Configuration:")
print("=" * 70)
print(json.dumps(inference_config, indent=2))

# Extract key parameters
model_name = training_config['model']['name']
num_labels = training_config['model']['num_labels']
batch_size = training_config['training']['batch_size']

print(f"\n✓ Configuration loaded")
print(f"  Model: {model_name}")
print(f"  Number of labels: {num_labels}")
print(f"  Training batch size: {batch_size}")

Training Configuration:
{
  "model": {
    "name": "distilbert-base-uncased",
    "pretrained": true,
    "hidden_size": 768,
    "num_labels": 2,
    "dropout_rate": 0.1
  },
  "training": {
    "epochs": 3,
    "batch_size": 2,
    "learning_rate": 2e-05,
    "weight_decay": 0.01,
    "warmup_steps": 500,
    "gradient_accumulation_steps": 4,
    "max_grad_norm": 1.0
  },
  "data": {
    "validation_split": 0.2,
    "test_split": 0.1,
    "max_length": 128,
    "dataset": "imdb"
  },
  "optimization": {
    "optimizer": "adam",
    "scheduler": "linear"
  },
  "output": {
    "model_path": "/models/trained_model",
    "metrics_path": "/models/metrics.json",
    "checkpoint_interval": 500
  }
}


Inference Configuration:
{
  "model": {
    "name": "distilbert-base-uncased",
    "model_path": "/models/trained_model",
    "max_length": 512
  },
  "serving": {
    "batch_size": 32,
    "num_workers": 4,
    "timeout": 30,
    "device": "cuda"
  },
  "output": {
    "format": "json",
    

## Section 4: Explore HuggingFace Model

Demonstrate model architecture and tokenization

In [4]:
# Explore the HuggingFace model backing the training/serving pipeline
from transformers import AutoTokenizer, AutoConfig, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"

print(f"Loading tokenizer for: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("  ✓ Tokenizer loaded")

print(f"\nLoading config for: {model_name}")
model_config = AutoConfig.from_pretrained(model_name, num_labels=2)
print("  ✓ Config loaded")

print("\nTokenizer Info:")
print(f"  Vocab size: {tokenizer.vocab_size}")
print(f"  Max position embeddings: {model_config.max_position_embeddings}")

print("\nModel Config:")
print(f"  Hidden size: {model_config.hidden_size}")
print(f"  Number of attention heads: {model_config.n_heads}")
print(f"  Number of hidden layers: {model_config.n_layers}")
print(f"  Intermediate size: {model_config.hidden_dim}")
print(f"  Number of labels: {model_config.num_labels}")

sample_text = "This is a great product!"
print(f"\nTokenization Example:")
print(f"  Text: {sample_text}")
encoded = tokenizer(sample_text)
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
print(f"  Tokens: {tokens}")
print(f"  Token IDs: {encoded['input_ids']}")
print(f"  Number of tokens: {len(encoded['input_ids'])}")

print("\nLoading full model to report real parameter counts (downloads pretrained weights)...")
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel Parameters:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size (fp32): ~{total_params * 4 / 1e6:.0f} MB")

print("\n✓ Model architecture explored")

Loading tokenizer for: distilbert-base-uncased


  ✓ Tokenizer loaded

Loading config for: distilbert-base-uncased
  ✓ Config loaded

Tokenizer Info:
  Vocab size: 30522
  Max position embeddings: 512

Model Config:
  Hidden size: 768
  Number of attention heads: 12
  Number of hidden layers: 6
  Intermediate size: 3072
  Number of labels: 2

Tokenization Example:
  Text: This is a great product!
  Tokens: ['[CLS]', 'this', 'is', 'a', 'great', 'product', '!', '[SEP]']
  Token IDs: [101, 2023, 2003, 1037, 2307, 4031, 999, 102]
  Number of tokens: 8

Loading full model to report real parameter counts (downloads pretrained weights)...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Model Parameters:
  Total parameters: 66,955,010
  Trainable parameters: 66,955,010
  Model size (fp32): ~268 MB

✓ Model architecture explored


## Section 5: Complete Pipeline Workflow

Demonstrate end-to-end sentiment classification: real preprocessing and tokenization, with real inference if a fine-tuned model is available locally

In [5]:
# Complete pipeline workflow: preprocessing -> tokenization -> inference
# Reuses the real preprocess_batch() from Section 2 and the real tokenizer from Section 4.
from src.models.inference import SentimentPredictor

print("=" * 70)
print("MLPIPELINE SENTIMENT CLASSIFICATION WORKFLOW")
print("=" * 70)

# Step 1: Raw Input Data
print("\nStep 1: Raw Input Data")
print("-" * 70)
raw_texts = [
    "This product is amazing! Highly recommended!",
    "Terrible quality, very disappointed.",
    "It's okay, nothing special.",
    "Love it! Best purchase ever!",
    "Waste of money, do not buy.",
]
for i, text in enumerate(raw_texts, 1):
    print(f"  {i}. {text}")

# Step 2: Data Preprocessing
print("\nStep 2: Data Preprocessing")
print("-" * 70)
preprocessed = preprocess_batch(raw_texts, clean=True)
for i, text in enumerate(preprocessed, 1):
    print(f"  {i}. {text}")

# Step 3: Tokenization
print("\nStep 3: Tokenization")
print("-" * 70)
batch_encoding = tokenizer(
    preprocessed, padding=True, truncation=True, max_length=512, return_tensors="pt"
)
print(f"  Tokenized {len(preprocessed)} texts with {model_name}")
print(f"  Input IDs shape: {tuple(batch_encoding['input_ids'].shape)}")
print(f"  Attention mask shape: {tuple(batch_encoding['attention_mask'].shape)}")

# Step 4: Model Inference -- needs a fine-tuned model on disk (produced by the training DAG)
print("\nStep 4: Model Inference")
print("-" * 70)
model_path = training_config["output"]["model_path"]
try:
    predictor = SentimentPredictor(model_path)
    predictions = predictor.predict_batch(raw_texts)

    print(f"\n  {'#':<2} {'Text':<40} {'Sentiment':<12} {'Confidence':<10}")
    print(f"  {'-' * 70}")
    for i, pred in enumerate(predictions, 1):
        text_short = pred["text"][:38]
        label = pred["label"].upper()
        conf = f"{pred['confidence']:.1%}"
        print(f"  {i:<2} {text_short:<40} {label:<12} {conf:<10}")

    print("\n✓ Pipeline workflow demonstrated end-to-end with a fine-tuned model")
except OSError as e:
    print(f"  No fine-tuned model found at {model_path!r}: {e}")
    print("  Run the `mlpipeline_test_training` (or `mlpipeline_training`) DAG first, then")
    print("  copy the model out of the `/models/trained_model` PV to run this step locally.")

MLPIPELINE SENTIMENT CLASSIFICATION WORKFLOW

Step 1: Raw Input Data
----------------------------------------------------------------------
  1. This product is amazing! Highly recommended!
  2. Terrible quality, very disappointed.
  3. It's okay, nothing special.
  4. Love it! Best purchase ever!
  5. Waste of money, do not buy.

Step 2: Data Preprocessing
----------------------------------------------------------------------
  1. this product is amazing highly recommended
  2. terrible quality very disappointed
  3. its okay nothing special
  4. love it best purchase ever
  5. waste of money do not buy

Step 3: Tokenization
----------------------------------------------------------------------
  Tokenized 5 texts with distilbert-base-uncased
  Input IDs shape: (5, 8)
  Attention mask shape: (5, 8)

Step 4: Model Inference
----------------------------------------------------------------------
  No fine-tuned model found at '/models/trained_model': Repo id must be in the form 'repo_nam

## Section 6: Deployment Architecture

Overview of Kubernetes deployment on kind-reunion

In [6]:
# Cell 6: Deployment Architecture Overview
import json

deployment_info = {
    "cluster": {
        "name": "kind-reunion",
        "namespace": "mlpipeline",
        "type": "Kubernetes (Kind)",
    },
    "components": {
        "orchestration": {
            "tool": "Apache Airflow 3.2.1rc1",
            "executor": "KubernetesPodOperator",
            "dags": [
                "training_dag.py - Training pipeline orchestration",
                "inference_dag.py - Batch inference pipeline",
                "test_training_dag.py - Quick training test on a 200-sample IMDB slice",
                "test_inference_dag.py - Test inference against the live serving endpoint",
                "gdp_etl_dag.py - GDP (BEA NIPA) ETL pipeline, independent of the sentiment pipeline"
            ]
        },
        "serving": {
            "framework": "FastAPI",
            "auth": "Keycloak OAuth2",
            "endpoints": [
                "POST /predict - Real-time sentiment prediction",
                "POST /predict-batch - Batch predictions",
                "GET /health - Health check",
                "GET /models - Model information"
            ]
        },
        "database": {
            "type": "PostgreSQL",
            "purpose": "Airflow metadata and state",
            "version": "16"
        },
        "authentication": {
            "provider": "Keycloak",
            "realm": "MLPipeline",
            "domain": "mlpipeline.duckdns.org"
        }
    },
    "features": [
        "Text preprocessing and cleaning",
        "Sentiment classification with DistilBERT",
        "Model versioning with DVC",
        "OAuth2 authentication",
        "TLS/HTTPS encryption",
        "Horizontal pod autoscaling",
        "Persistent volume management"
    ]
}

print("MLPipeline Kubernetes Deployment Architecture")
print("=" * 70)
print(json.dumps(deployment_info, indent=2))

print("\n\nQuick Start Commands:")
print("-" * 70)
commands = [
    ("Deploy to Kubernetes", "./scripts/deploy.sh"),
    ("Setup Keycloak OAuth", "./scripts/setup-keycloak.sh"),
    ("Port-forward Airflow", "kubectl port-forward -n mlpipeline svc/airflow-webserver 8080:8080"),
    ("Port-forward FastAPI", "kubectl port-forward -n mlpipeline svc/mlpipeline-serving 8000:8000"),
    ("View pods", "kubectl get pods -n mlpipeline"),
    ("View logs", "kubectl logs -n mlpipeline -f deployment/airflow-scheduler"),
]

for description, command in commands:
    print(f"\n{description}:")
    print(f"  $ {command}")

print("\n✓ Deployment architecture outlined")

MLPipeline Kubernetes Deployment Architecture
{
  "cluster": {
    "name": "kind-reunion",
    "namespace": "mlpipeline",
    "type": "Kubernetes (Kind)"
  },
  "components": {
    "orchestration": {
      "tool": "Apache Airflow 3.2.1rc1",
      "executor": "KubernetesPodOperator",
      "dags": [
        "training_dag.py - Training pipeline orchestration",
        "inference_dag.py - Batch inference pipeline",
        "test_training_dag.py - Quick training test on a 200-sample IMDB slice",
        "test_inference_dag.py - Test inference against the live serving endpoint",
        "gdp_etl_dag.py - GDP (BEA NIPA) ETL pipeline, independent of the sentiment pipeline"
      ]
    },
    "serving": {
      "framework": "FastAPI",
      "auth": "Keycloak OAuth2",
      "endpoints": [
        "POST /predict - Real-time sentiment prediction",
        "POST /predict-batch - Batch predictions",
        "GET /health - Health check",
        "GET /models - Model information"
      ]
    },
    "

## Next Steps

1. **Deploy to Kubernetes**: Run `./scripts/deploy.sh` from the repository root
2. **Configure Keycloak**: Execute `./scripts/setup-keycloak.sh`
3. **Access Airflow**: Navigate to https://mlpipeline.duckdns.org/airflow
4. **Test FastAPI**: Use https://mlpipeline.duckdns.org/api/docs
5. **Submit Training Job**: Trigger `mlpipeline_training` DAG from Airflow UI

## Resources

- **Documentation**: See README.md and DEPLOYMENT.md
- **Keycloak Setup**: See KEYCLOAK_SETUP.md  
- **Source Code**: Explore `src/`, `dags/`, `serving/` directories
- **Configuration**: Check `configs/` for model and pipeline settings
- **Tests**: Run tests with `pytest tests/`

---

**MLPipeline v1.0.0** - NLP ML Pipeline on Kubernetes with OAuth Authentication